# Basic Jax Grad Methods

## Lesson Goals:

By the end of this lesson, you will understand the main `grad`-related functions that `jax` exposes, and when to use them. 

**Note**: Autodiff and the various optimizations one can implement are a very deep area of study. Our aim here is to give you a high-level overview and teach you enough to be dangerous.

## Core Concepts:

- `grad` for getting the gradient with respect to some arguments
- `value_and_grad`, for getting the result of the function evaluation, and the gradient with respect to some arguments

## Related Notebooks

- [Grad Manipulations Notebook](./exe_08_grad_intermediate.ipynb) - gradient manipulation methods
- [Advanced Autodiff](./exe_10_grad_advanced.ipynb) - custom derivatives

## Additional Reading:

- [Jax Autodiff Cookbook](https://docs.jax.dev/en/latest/notebooks/autodiff_cookbook.html)

- [Bonus - JVP and VJP](./bonus_autodiff_jvp_vjp.ipynb)

## Concepts In action:

- [Logistic Regression](../case_studies/binary-classification/logistic_regression.ipynb)

In [ ]:
import jax
import jax.numpy as jnp
from jax import vmap
from jax import grad, value_and_grad
from jax import random

import matplotlib.pyplot as plt

# Vanilla Gradient: `jax.grad`

- `jax.grad` is the simplest form of taking a gradient in jax, and is what you will probably use 90% of the time.
- with `jax.grad` you can take the gradient arbitrarily many times. Let's explore this
- it accepts a function and returns a new function! 

## `grad` signature

Assume `CT a` refers to the "space" that the gradients of `a` live in$^\text{Note}$Say the `grad` function has the following signature:

`grad :: (a -> b) -> (a -> (CT a))`

as a function that accepts a function and returns a new function that accepts a vector, along which to take the gradient, and returns the gradient.

$^\text{Note}$ see [Bonus - JVP and VJP](./bonus_autodiff_jvp_vjp.ipynb) if you're really interested, but it's not necessary. 

---

Let's now consider a simple example:


In [ ]:
x = jnp.linspace(-2, 2, 100)

def arbitrary_gradients():
    """
    TODO:
        take the gradient of `fx` up to the third degree! You will find `vmap` useful
    """
    fx = lambda x: 4*x**3 + 3*x**2
    fn_d1_dx = ...
    fn_d2_dx = ...
    fn_d3_dx = ...

    ys = fx(x)
    d1_dx = fn_d1_dx(x)
    d2_dx = fn_d2_dx(x)
    d3_dx = fn_d3_dx(x)
    
    plt.plot(x, ys, label="f(x)=$4x^3 + 3x^2$")
    plt.plot(x, d1_dx, label="f'(x)=$12x^2 + 6x$")
    plt.plot(x, d2_dx, label="f$^{(2)}$=$24x + 6$")
    plt.plot(x, d3_dx, label="f$^{(3)}$=24")
    plt.legend()
    plt.show()

arbitrary_gradients()

## `value_and_grad` signature

The gradient was useful, but other times we want to get the value of the function evaluation and the gradient simultaneously, which is where `value_and_grad` comes in. `value_and_grad` has the signature of:

`value_and_grad :: (a -> b) -> (a -> (b, CT a))`

i.e. it is a function that accepts a function and returns a new function (that returns a tuple):

- our `b`, the result of the function evaluation
- our `CT a` is the gradient of the input, in the "gradient" space.

---

Let's now consider a simple example (note: the additional arguments are present in both `grad` and `value_and_grad`)

In [ ]:
W = jnp.asarray([jnp.pi - 0.55, 2 * jnp.pi - 0.55])
b = jnp.asarray([0.5])

def simple_loss(W, b):
    W_prime = jnp.asarray([jnp.pi, 2 * jnp.pi])
    target = jnp.cos(W_prime)
    diff = target - (jnp.cos(W) + b)
    return jnp.sum(jnp.sqrt(diff ** 2))

## Value And Grad of a specific argument

Gradient with respect to the 0-th argument of `simple_loss`.

NOTE: if argnums is not passed in, jax will default to the derivative wrt. the first argument of the `simple_loss` function

In [ ]:
loss_val, b_grad = value_and_grad(simple_loss, argnums=0)(W, b)
print(f'{b_grad=}')
print(f'{loss_val=}')

## Value And Grad w.r.t. multiple arguments

Unsurprisingly, we can also take the gradient with respect to both arguments at once

In [ ]:
loss_val, (W_grad, b_grad) = value_and_grad(simple_loss, (0, 1))(W, b)
print(f'{W_grad=}')
print(f'{b_grad=}')
print(f'{loss_val=}')

## Getting the `grad` of a structure

- `grad` doesn't require that we pass in jax arrays - we can theoretically pass in any container type, so long as we "register" it with `jax` (more on this in the [pytrees notebook](./exe_09_pytrees.ipynb))

In [ ]:

def simple_loss_w_dict(param_dict):
    W_prime = jnp.asarray([jnp.pi, 2 * jnp.pi])
    target = jnp.cos(W_prime)
    diff = target - (jnp.cos(param_dict["W"]) + param_dict["b"])
    return jnp.sum(jnp.sqrt(diff ** 2))

loss_val, grads = value_and_grad(simple_loss_w_dict)({"W": W, "b": b})
print(f'{grads=}')
print(f'{loss_val=}')

# Conclusion

This one is relatively short, because `grad` and `value_and_grad` will cover most of what you need to know and they are (conveniently) not too difficult to conceptualize!